# Stage 1 — Exploratory Data Analysis

Reads the raw parquet landed in stage 0 directly from S3 with **DuckDB**, instead of pandas.

Why: DuckDB pushes filters/aggregations down and streams from S3 without materializing the full
~51M-row long table in memory — the same shape of problem that caused the stage-0 OOM. Only small,
already-aggregated results get pulled into pandas for plotting.

**Cost note**: this is CPU/IO-light. The current `ml.m5.xlarge` (left running from stage 0) is
comfortably enough, and this approach would also work fine on `ml.t3.medium` if you'd rather
downsize — feel free to do that via Stop → Edit instance type → Start before running this.

In [ ]:
%pip install -q duckdb matplotlib

In [ ]:
import boto3
import duckdb
import matplotlib.pyplot as plt

BUCKET = "<your-bucket>"
RAW_PREFIX = "ts-forecast-demo/raw/electricity"
CURATED_PREFIX = "ts-forecast-demo/curated"
RAW_GLOB = f"s3://{BUCKET}/{RAW_PREFIX}/year=*/*.parquet"

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Reuse the notebook instance's own execution-role credentials (incl. session token)
# instead of hardcoding keys.
creds = boto3.Session().get_credentials().get_frozen_credentials()
region = boto3.Session().region_name or "us-east-1"
con.execute(f"SET s3_region='{region}';")
con.execute(f"SET s3_access_key_id='{creds.access_key}';")
con.execute(f"SET s3_secret_access_key='{creds.secret_key}';")
if creds.token:
    con.execute(f"SET s3_session_token='{creds.token}';")

## 1. Sanity check: shape and date range

In [ ]:
con.sql(f"""
    SELECT
        count(*) AS n_rows,
        count(DISTINCT client_id) AS n_clients,
        min(timestamp) AS min_ts,
        max(timestamp) AS max_ts
    FROM read_parquet('{RAW_GLOB}', hive_partitioning=1)
""").show()

## 2. Per-client activation profile

The dataset docs note that clients added after 2011 show zero consumption before their real start —
not a missing-data gap, a "not yet a customer" period. We need each client's *actual* first non-zero
reading to avoid training on fake leading zeros.

In [ ]:
client_profile = con.sql(f"""
    SELECT
        client_id,
        min(timestamp) FILTER (WHERE kwh > 0) AS first_active_ts,
        max(timestamp) AS last_ts,
        count(*) AS n_readings,
        sum(CASE WHEN kwh = 0 THEN 1 ELSE 0 END) AS n_zero_readings,
        avg(kwh) AS mean_kwh,
        max(kwh) AS max_kwh
    FROM read_parquet('{RAW_GLOB}', hive_partitioning=1)
    GROUP BY client_id
""").df()

client_profile["active_days"] = (client_profile["last_ts"] - client_profile["first_active_ts"]).dt.days
client_profile["zero_fraction"] = client_profile["n_zero_readings"] / client_profile["n_readings"]
client_profile.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(client_profile["active_days"], bins=40)
axes[0].set_title("Active history length (days) per client")
axes[1].hist(client_profile["zero_fraction"], bins=40)
axes[1].set_title("Fraction of zero readings per client")
plt.tight_layout()

## 3. Filtering rule for modeling

Candidate rule, tune after looking at the histograms above: keep clients with at least 2 full years
of active (post-activation) history, so every model sees at least two full annual seasonal cycles.

In [ ]:
MIN_ACTIVE_DAYS = 730
valid_clients = client_profile[client_profile["active_days"] >= MIN_ACTIVE_DAYS]
print(f"{len(valid_clients)} / {len(client_profile)} clients pass the {MIN_ACTIVE_DAYS}-day filter")

## 4. Look at a few individual series

Pull only a handful of client_ids -- a query filtered this way is small regardless of the
underlying table size.

In [ ]:
sample_ids = valid_clients["client_id"].sample(4, random_state=42).tolist()
id_list = ", ".join(f"'{c}'" for c in sample_ids)

sample_df = con.sql(f"""
    SELECT timestamp, client_id, kwh
    FROM read_parquet('{RAW_GLOB}', hive_partitioning=1)
    WHERE client_id IN ({id_list})
""").df()

fig, ax = plt.subplots(figsize=(12, 4))
for cid, grp in sample_df.groupby("client_id"):
    daily = grp.set_index("timestamp")["kwh"].resample("1D").sum()
    ax.plot(daily.index, daily.values, label=cid, alpha=0.8)
ax.legend()
ax.set_title("Daily consumption, 4 sampled clients")

## 5. Seasonality: hour-of-day and day-of-week

Computed as an aggregation pushed down to DuckDB across the full table -- cheap even though
the underlying data is ~51M rows, since we never materialize row-level data in pandas.

In [ ]:
con.register("valid_clients_view", valid_clients[["client_id"]])

seasonality = con.sql(f"""
    SELECT
        extract('hour' FROM timestamp) AS hour_of_day,
        extract('dow' FROM timestamp) AS day_of_week,
        avg(kwh) AS mean_kwh
    FROM read_parquet('{RAW_GLOB}', hive_partitioning=1) r
    JOIN valid_clients_view v USING (client_id)
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

pivot = seasonality.pivot(index="hour_of_day", columns="day_of_week", values="mean_kwh")
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, aspect="auto", origin="lower")
ax.set_xlabel("day_of_week (0=Sun)")
ax.set_ylabel("hour_of_day")
ax.set_title("Mean kWh by hour x weekday, across valid clients")
fig.colorbar(im)

## 6. Persist EDA outputs for later stages

`client_profile` (with the validity flag) becomes an early candidate for the Feature Store in
stage 3 -- static, per-entity metadata used to filter/join at training time.

In [ ]:
client_profile["is_valid"] = client_profile["client_id"].isin(valid_clients["client_id"])

out_path = "client_metadata.parquet"
client_profile.to_parquet(out_path, index=False)

s3 = boto3.client("s3")
s3.upload_file(out_path, BUCKET, f"{CURATED_PREFIX}/{out_path}")
print(f"Uploaded to s3://{BUCKET}/{CURATED_PREFIX}/{out_path}")

## Findings / decisions for Stage 2 (modeling)

- Fill in after running: how many clients pass the 730-day filter?
- Confirm whether 2011 should be dropped entirely (many clients not yet active) vs. handled via
  the per-client `first_active_ts` cutoff instead.
- Note the dominant seasonality (daily cycle amplitude vs. weekday/weekend effect) -- this informs
  which covariates DeepAR/TFT should be given (hour-of-day, day-of-week, is_weekend).
- Decide raw 15-min vs. resampled hourly granularity for training -- directly trades off sequence
  length (compute cost) against resolution.